hey there! this code is written and maintained by [amilleah](https://github.com/amilleah/)!

its purpose is to encode writing style into PCA space using linguistic features. 

if modified, please indicate in this cell with your changes:

```txt
modified by @amilleah on 04-18-2026:
    - add .txt support
```

In [ ]:
#---- get sorted data
from pathlib import Path
import re

def index_key(p):       #--- expected file format: ii_{date}_{slug}.txt
    match = re.match(r'^(\d+)', p.stem)
    return int(match.group(1)) if match else -1

CORPUS = "./corpus/"
EXCLUDE = []
DATA = {p.stem: p.read_text() for p in sorted(Path(CORPUS).glob("*.txt"), key=index_key) if p.stem not in EXCLUDE}

print(DATA)

In [ ]:
import nltk
nltk.download('punkt_tab', quiet=True)

rows = []
for name, text in DATA.items():
    # normalize line separators before tokenizing
    clean = text.replace('\u2028', ' ').replace('\n', ' ')
    for sent in nltk.sent_tokenize(clean):
        rows.append({'newsletter': name, 'sentence': sent.strip()})

import pandas as pd
df = pd.DataFrame(rows)
print(df.head())

In [ ]:
#---- NLP models (expects you've run `uv run python -m spacy download en_core_web_sm` or downloaded a comparable model)
import spacy
from sentence_transformers import SentenceTransformer

nlp   = spacy.load('en_core_web_sm')
sbert = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
#---- get SBERT embeddings
print("extracting sentence embeddings...")
embeddings = sbert.encode(df["sentence"].tolist(), show_progress_bar=True, batch_size=64)

In [ ]:
from collections import Counter
from tqdm import tqdm

def get_part_of_speech(doc):
    total = sum(1 for t in doc if not t.is_space)
    counts = Counter(t.pos_ for t in doc if not t.is_space)
    return {k: v / total for k, v in counts.items()} if total else {}

def get_pronouns(doc):
    total = sum(1 for t in doc if not t.is_space)
    if not total:
        return {}
    kinds = Counter()
    for t in doc:
        if t.pos_ == "PRON":
            person = t.morph.get("Person", ["?"])[0]   # 1, 2, 3
            number = t.morph.get("Number", ["?"])[0]   # sing, plur
            kinds[f"pron_{person}{number}"] += 1
    return {k: v / total for k, v in kinds.items()}

def get_dependencies(doc):
    total = len(doc)
    counts = Counter(t.dep_ for t in doc)
    return {k: v / total for k, v in counts.items()} if total else {}

def get_clauses(doc):
    clause_deps = {"relcl", "advcl", "acl", "ccomp", "xcomp"}
    return sum(1 for t in doc if t.dep_ in clause_deps)

def get_punctuation(doc):
    total = len(doc)
    counts = Counter(t.text for t in doc if t.pos_ == "PUNCT")
    return {k: v / total for k, v in counts.items()} if total else {}

def get_ttr(text):
    tokens = text.lower().split()
    return len(set(tokens)) / len(tokens) if tokens else 0.0

#---- encoding the feature set per sentence, per document
def encode_features(df):
    records = []
    for _, letter in tqdm(df.groupby('newsletter'), total=df['newsletter'].nunique()):
        sentence_idx = 0
        n_sents = len(letter)
        for _, row in letter.iterrows():
            doc = nlp(row['sentence'])
            records.append({
                **{f"pos_{k}":  v for k, v in get_part_of_speech(doc).items()},
                **get_pronouns(doc),
                **{f"dep_{k}":  v for k, v in get_dependencies(doc).items()},
                **{f"punc_{k}": v for k, v in get_punctuation(doc).items()},
                "clauses":           get_clauses(doc),
                "sentence_len":      len(row['sentence']),
                "words":             len(row['sentence'].split()),
                "ttr":               get_ttr(row['sentence']),
                "sentence_position": sentence_idx / max(n_sents - 1, 1),
            })
            sentence_idx += 1
    return pd.DataFrame(records).fillna(0).reset_index(drop=True)

features = encode_features(df)

In [ ]:
import numpy as np
from itertools import combinations
from scipy.linalg import svd
from scipy.stats import permutation_test
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

#---- assemble feat_df
sbert_df   = pd.DataFrame(embeddings, columns=[f"sbert_{i}" for i in range(embeddings.shape[1])])
feat_df    = pd.concat([df[['newsletter', 'sentence']].reset_index(drop=True), features, sbert_df], axis=1)

SBERT_COLS = [c for c in feat_df.columns if c.startswith("sbert_")]
doc_order  = list(df['newsletter'].unique())

#---- feature cols: frequency filter then correlation filter
mask      = (feat_df[features.columns] != 0).mean() > 0.2
corr      = feat_df[features.columns[mask]].corr().abs()
upper     = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
drop_cols = [c for c in upper.columns if any(upper[c] > 0.7)]
COLUMNS   = [c for c in features.columns[mask] if c not in drop_cols]
print(f"COLUMNS: {len(COLUMNS)}")

#---- core functions
def principal_angles(A, B):
    Q_A, _ = np.linalg.qr(A.T)
    Q_B, _ = np.linalg.qr(B.T)
    _, s, _ = svd(Q_A.T @ Q_B)
    return np.degrees(np.arccos(np.clip(s, -1, 1)))

def vaf(data, components):
    proj = data @ components.T @ components
    return 1 - np.sum((data - proj)**2) / np.sum(data**2)

#---- K via bootstrap subspace stability
scaled    = StandardScaler().fit_transform(feat_df[COLUMNS].values.astype(float))
N_BOOT    = 250
MAX_K     = min(8, len(COLUMNS) // 2)
STABLE_THRESHOLD = 25.0

full_pca  = PCA(n_components=MAX_K).fit(scaled)
boot_max  = {k: [] for k in range(1, MAX_K + 1)}

for _ in range(N_BOOT):
    idx      = np.random.choice(len(scaled), len(scaled), replace=True)
    k_eff    = min(MAX_K, idx.shape[0], scaled.shape[1])
    boot_pca = PCA(n_components=k_eff).fit(scaled[idx])
    for k in range(1, k_eff + 1):
        pa = principal_angles(full_pca.components_[:k], boot_pca.components_[:k])
        boot_max[k].append(float(pa.max()))

print(f"{'k':>3}  {'p95_max_angle':>14}  {'stable':>7}")
K = 0
for k, angles in boot_max.items():
    p95    = float(np.percentile(angles, 95))
    stable = p95 < STABLE_THRESHOLD
    print(f"{k:>3}  {p95:>14.1f}°  {'yes' if stable else 'no --- stop':>10}")
    if not stable:
        break
    K = k
K = max(K, 1)
print(f"\nK={K}")

components = full_pca.components_[:K]

#---- N×N pairwise cross-newsletter VAF
N          = len(doc_order)
rand_comps = {
    pid: np.linalg.qr(np.random.randn(len(COLUMNS), K))[0].T
    for pid in doc_order
}

post_syn_data, post_syn_comps = {}, {}
post_sem_data, post_sem_comps = {}, {}
for pid in doc_order:
    m = feat_df['newsletter'] == pid
    syn_mat = StandardScaler().fit_transform(feat_df[m][COLUMNS].values.astype(float))
    sem_mat = StandardScaler().fit_transform(feat_df[m][SBERT_COLS].values.astype(float))
    k_eff   = min(K, syn_mat.shape[0] - 1, syn_mat.shape[1])
    post_syn_data[pid]  = syn_mat
    post_syn_comps[pid] = PCA(n_components=k_eff).fit(syn_mat).components_
    post_sem_data[pid]  = sem_mat
    post_sem_comps[pid] = PCA(n_components=k_eff).fit(sem_mat).components_

mat_syn  = np.full((N, N), np.nan)
mat_sem  = np.full((N, N), np.nan)
mat_rand = np.full((N, N), np.nan)

for j, pid_j in enumerate(doc_order):
    syn_j      = post_syn_data[pid_j]
    sem_j      = post_sem_data[pid_j]
    own_syn    = vaf(syn_j, post_syn_comps[pid_j])
    own_sem    = vaf(sem_j, post_sem_comps[pid_j])
    for i, pid_i in enumerate(doc_order):
        if own_syn > 0:
            mat_syn[i, j]  = vaf(syn_j, post_syn_comps[pid_i]) / own_syn
            mat_rand[i, j] = vaf(syn_j, rand_comps[pid_i])     / own_syn
        if own_sem > 0:
            mat_sem[i, j]  = vaf(sem_j, post_sem_comps[pid_i]) / own_sem

off = ~np.eye(N, dtype=bool)
vaf_syn_off  = mat_syn[off]
vaf_sem_off  = mat_sem[off]
vaf_rand_off = mat_rand[off]

def statistic(x, y): return np.median(x) - np.median(y)
res = permutation_test((vaf_syn_off, vaf_sem_off), statistic, permutation_type='samples', n_resamples=10000)

print(f"\nposts: {N},  pairs: {N*(N-1)}")
print(f"syntactic VAF: {np.nanmedian(vaf_syn_off):.3f}  semantic: {np.nanmedian(vaf_sem_off):.3f}  random: {np.nanmedian(vaf_rand_off):.3f}")
print(f"permutation p={res.pvalue:.4f}")

In [ ]:
#---- print PCA loading summaries

K = 1

#---- project onto the bootstrap-stable global axes
coords = full_pca.transform(scaled)[:, :K]
for i in range(K):
    feat_df[f"pc{i+1}"] = coords[:, i]

#---- feature labels
POS_LABELS = {
    "ADJ": "Adjectives",
    "ADP": "Prepositions",
    "ADV": "Adverbs",
    "AUX": "Auxiliary verbs",
    "CCONJ": "Coordinating conjunctions",
    "DET": "Determiners",
    "NOUN": "Nouns",
    "NUM": "Numbers",
    "PART": "Particles",
    "PRON": "Pronouns",
    "PROPN": "Proper nouns",
    "PUNCT": "Punctuation",
    "SCONJ": "Subordinating conjunctions",
    "VERB": "Verbs",
}

PRON_LABELS = {
    "1Sing": "First-person singular pronouns",
    "1Plur": "First-person plural pronouns",
    "2?": "Second-person pronouns",
    "3Sing": "Third-person singular pronouns",
    "?Sing": "Singular pronouns",
    "??": "Unspecified pronouns",
}

DEP_LABELS = {
    "ROOT": "Root dependencies",
    "acl": "Clausal modifiers",
    "acomp": "Adjectival complements",
    "advcl": "Adverbial clauses",
    "amod": "Adjectival modifiers",
    "attr": "Attribute dependencies",
    "aux": "Auxiliary dependencies",
    "ccomp": "Clausal complements",
    "compound": "Compound modifiers",
    "conj": "Conjunct dependencies",
    "dobj": "Direct objects",
    "mark": "Clause markers",
    "neg": "Negations",
    "npadvmod": "Noun-phrase adverbial modifiers",
    "nsubj": "Nominal subjects",
    "poss": "Possessive modifiers",
    "prt": "Particle dependencies",
    "relcl": "Relative clauses",
    "xcomp": "Open clausal complements",
}

PUNC_LABELS = {
    ",": "Commas",
    ".": "Periods",
    "-": "Dashes",
}

FEATURE_LABELS = {
    "clauses": "Clause count",
    "sentence_len": "Character count",
    "sentence_position": "Sentence position",
    "ttr": "Type-token ratio",
    "words": "Word count",
}

def make_label(col):
    if col.startswith("pos_"):
        tag = col[4:]
        return POS_LABELS.get(tag, tag.title())
    if col.startswith("pron_"):
        tag = col[5:]
        return PRON_LABELS.get(tag, f"Pronouns {tag}")
    if col.startswith("dep_"):
        dep = col[4:]
        return DEP_LABELS.get(dep, f"{dep} dependencies")
    if col.startswith("punc_"):
        mark = col[5:]
        return PUNC_LABELS.get(mark, f"{mark} punctuation")
    return FEATURE_LABELS.get(col, col.replace("_", " ").title())

feat_label_map = {col: make_label(col) for col in COLUMNS}

#---- top loadings from axes
TOP_N = 20
variance_explained = [float(full_pca.explained_variance_ratio_[i]) for i in range(K)]

print(f"K={K}")
print(f"variance explained: {' '.join(f'PC{i+1}={variance_explained[i]:.1%}' for i in range(K))}  total={sum(variance_explained):.1%}")

for i in range(K):
    loading_pairs = [
        (feat_label_map[col], float(components[i][COLUMNS.index(col)]))
        for col in COLUMNS
    ]
    top_loadings = sorted(loading_pairs, key=lambda item: abs(item[1]), reverse=True)[:TOP_N]

    print(f"\nPC{i+1} top {TOP_N} loadings by absolute value")
    for label, value in top_loadings:
        print(f"  {label}: {value:+.4f}")